# LoRA fine-tuning on CPU in a few minutes

The whole SageMaker fine-tuning loop, end to end, on CPU instances only, so it runs in any account without
waiting for GPU capacity:

1. a training job fine-tunes **Qwen3-0.6B** with a LoRA adapter (about 20 steps) and evaluates the base and the
   fine-tuned model on held-out examples, in the same job
2. you read the metrics and see what changed
3. optionally, a CPU real-time endpoint serves the base model + adapter

**The task.** Turn a free-text restaurant booking request into strict JSON:

`"hey! Dana - four ppl friday 7pm if possible"` → `{"name": "Dana", "party_size": 4, "day": "Friday", "time": "19:00"}`

The data is generated in code (`scripts/booking_data.py`), so nothing is downloaded except the model. The base
model gets the same instruction, and usually gets the values roughly right but not the exact format (it wraps
the JSON in a markdown code block, keeps "friday" lowercase, writes "7pm"). A few LoRA steps teach the format.

The model is small and the run short on purpose: this notebook shows the mechanics (training job, adapter,
metrics, endpoint). The same pattern scales to GPU instances and larger models in the other labs.

In [ ]:
%pip install -q "sagemaker==3.21.0"

In [ ]:
import json
import time

import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
region = sess.boto_region_name
try:
    role = get_execution_role()
except ValueError:
    role = boto3.client("iam").get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]
bucket = sess.default_bucket()
prefix = "/".join(p for p in (sess.default_bucket_prefix, "cpu-lora-quickstart") if p)
sm = boto3.client("sagemaker", region_name=region)
s3 = boto3.client("s3", region_name=region)
print(f"role:   {role}\nregion: {region}\noutput: s3://{bucket}/{prefix}/")

## The data

In [ ]:
import sys

sys.path.insert(0, "scripts")
from booking_data import INSTRUCTION, make_splits

train, test = make_splits(n_train=400, n_test=30)
print(f"instruction: {INSTRUCTION}\n{len(train)} training / {len(test)} test examples\n")
for ex in train[:3]:
    print(f"{ex['request']}\n  -> {ex['target']}")

## Train and evaluate on a CPU instance

One training job does everything (`scripts/train.py`): it scores the base model, trains the adapter, scores the
fine-tuned model, and writes `adapter/` and `metrics.json`. The output is stored uncompressed so the endpoint
below can use it directly.

The job runs on the first instance type in `INSTANCE_TYPES` that has capacity (`instance_preferences`).
SageMaker rejects the whole list if any type in it has no training quota, so types without quota are dropped
first.

In [ ]:
from sagemaker.core.shapes import InstancePreference
from sagemaker.core.training.configs import Compute, OutputDataConfig, SourceCode, StoppingCondition
from sagemaker.train import ModelTrainer

STEPS = 20
INSTANCE_TYPES = ["ml.c7i.4xlarge", "ml.m7i.4xlarge", "ml.c6i.4xlarge", "ml.m6i.4xlarge", "ml.m5.4xlarge"]

try:
    quotas = {q["QuotaName"]: q["Value"]
              for page in boto3.client("service-quotas", region_name=region).get_paginator("list_service_quotas")
              .paginate(ServiceCode="sagemaker") for q in page["Quotas"]}
    INSTANCE_TYPES = [t for t in INSTANCE_TYPES if quotas.get(f"{t} for training job usage", 0) >= 1] or INSTANCE_TYPES
except Exception as e:  # role cannot read Service Quotas: send the full list
    print(f"quota check skipped: {e}")
print(f"instance types: {INSTANCE_TYPES}")

trainer = ModelTrainer(
    training_image=f"763104351884.dkr.ecr.{region}.amazonaws.com/pytorch-training:2.8-cpu-py312",
    source_code=SourceCode(source_dir="scripts", requirements="requirements.txt", entry_script="train.py"),
    hyperparameters={"steps": STEPS, "model_id": "Qwen/Qwen3-0.6B"},
    compute=Compute(instance_preferences=[InstancePreference(instance_type=t) for t in INSTANCE_TYPES],
                    instance_count=1, volume_size_in_gb=30),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=1800),
    output_data_config=OutputDataConfig(s3_output_path=f"s3://{bucket}/{prefix}/jobs", compression_type="NONE"),
    base_job_name="cpu-lora-quickstart",
    role=role,
    sagemaker_session=sess,
)
trainer.train(wait=False)
job_name = trainer._latest_training_job.training_job_name
print(f"submitted: {job_name}")

### Wait for the job

The job runs in SageMaker independently of this notebook. If the kernel restarts, re-run the setup cell and this
one: it finds the latest quickstart job.

In [ ]:
job_name = globals().get("job_name") or sm.list_training_jobs(
    NameContains="cpu-lora-quickstart", SortBy="CreationTime", SortOrder="Descending", MaxResults=1
)["TrainingJobSummaries"][0]["TrainingJobName"]
last = None
while True:
    job = sm.describe_training_job(TrainingJobName=job_name)
    line = f"{job['TrainingJobStatus']}/{job['SecondaryStatus']}"
    if line != last:
        print(time.strftime("%H:%M:%S"), line, flush=True)
        last = line
    if job["TrainingJobStatus"] in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(15)
if job["TrainingJobStatus"] != "Completed":
    raise RuntimeError(job.get("FailureReason", job["TrainingJobStatus"]))
for t in job["SecondaryStatusTransitions"]:
    end = t.get("EndTime") or job["TrainingEndTime"]
    print(f"  {t['Status']:<12} {(end - t['StartTime']).total_seconds():5.0f} s")
rc = job["ResourceConfig"]
print(f"instance: {rc.get('SelectedInstanceType') or rc.get('InstanceType')}, billed {job['BillableTimeInSeconds']} s")

## Results

In [ ]:
model_prefix = job["ModelArtifacts"]["S3ModelArtifacts"].rstrip("/") + "/"
mb, mk = model_prefix[5:].split("/", 1)
metrics = json.loads(s3.get_object(Bucket=mb, Key=mk + "metrics.json")["Body"].read())

print(f"{'':10} {'exact':>6} " + " ".join(f"{f:>10}" for f in ("name", "party_size", "day", "time")))
for which in ("base", "finetuned"):
    m = metrics[which]
    print(f"{which:10} {m['exact_match']:6.0%} " + " ".join(f"{m['acc_' + f]:10.0%}" for f in ("name", "party_size", "day", "time")))
print(f"\n{metrics['steps']} steps, {metrics['seconds_training']} s training, {metrics['seconds_total']} s in the script\n")
for ex in metrics["examples"][:2]:
    print(f"request:   {ex['request']}\nbase:      {ex['base']!r}\nfinetuned: {ex['finetuned']}\n")

## Optional: serve it from a CPU endpoint

The endpoint loads the base model from the Hugging Face Hub and the adapter from the training job's output,
merges them at startup (`serving/inference.py`), and answers JSON requests. It takes a few minutes to start and
bills per hour until you delete it (last cell).

In [ ]:
for name in ("inference.py", "requirements.txt"):
    s3.upload_file(f"serving/{name}", mb, f"{mk}code/{name}")

endpoint = f"cpu-lora-quickstart-{time.strftime('%Y%m%d-%H%M%S')}"
sm.create_model(ModelName=endpoint, ExecutionRoleArn=role, PrimaryContainer={
    "Image": f"763104351884.dkr.ecr.{region}.amazonaws.com/pytorch-inference:2.6-cpu-py312",
    "ModelDataSource": {"S3DataSource": {"S3Uri": model_prefix, "S3DataType": "S3Prefix", "CompressionType": "None"}},
    "Environment": {"SAGEMAKER_PROGRAM": "inference.py", "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
                    "SAGEMAKER_MODEL_SERVER_WORKERS": "1"},
})
sm.create_endpoint_config(EndpointConfigName=endpoint, ProductionVariants=[{
    "VariantName": "AllTraffic", "ModelName": endpoint, "InstanceType": "ml.m5.xlarge", "InitialInstanceCount": 1}])
sm.create_endpoint(EndpointName=endpoint, EndpointConfigName=endpoint)
sm.get_waiter("endpoint_in_service").wait(EndpointName=endpoint, WaiterConfig={"Delay": 20, "MaxAttempts": 60})
print(f"in service: {endpoint}")

In [ ]:
runtime = boto3.client("sagemaker-runtime", region_name=region)
for request in ["hey! Dana - four ppl friday 7pm if possible",
                "Could I get a table for 2 on sunday at 12:30 PM? It's Hiro."]:
    resp = runtime.invoke_endpoint(EndpointName=endpoint, ContentType="application/json",
                                   Body=json.dumps({"request": request}))
    print(request, "->", json.loads(resp["Body"].read())["booking"])

In [ ]:
# Clean up the endpoint (it bills per hour while it exists)
sm.delete_endpoint(EndpointName=endpoint)
sm.delete_endpoint_config(EndpointConfigName=endpoint)
sm.delete_model(ModelName=endpoint)
print("deleted", endpoint)